# 🔬 Virtual Lab 4: Running Llama Models on Ollama

<div style="border: 2px solid #4CAF50; padding: 15px; border-radius: 10px; background-color: #f4f4f4;">
    
### 🚀 **Platform**  
**Ollama**

### 🏷️ **Models & Sizes**  
- **llama3.2:1b** →  **1.3GB**  
- **llama3.2:3b** →  **2.0GB**  
- **llama3.2:3b-instruct-fp16** →  **6.4GB**  

### 🛠️ **Frameworks Used**  
- **LlamaIndex**  
- **LangChain / LangGraph**  

</div>

## Model Performance and Memory Requirements

### **Model: llama3.2:3b**  
- Using Ollama, it takes roughly **10 minutes** to complete.  
- Requires **8GB minimum main memory**.  

### **Model: llama3.2:1b**  
- Using Ollama, it **doesn't work for creating AI Agents**.  
- Requires **8GB minimum main memory**.  

### **Model: llama3.2:3b-instruct-fp16**  
- Using Ollama, it requires **at least 32GB main memory** to work.  
- Still, it is **very slow** (generates **one token per second** for output).  

## Installation and Setup

In [ ]:
!pip install llama-index
!pip install llama-index-llms-ollama
!pip install llama-index-llms-replicate
!pip install llama-index-embeddings-huggingface
!pip install llama-parse
!pip install replicate

In [ ]:
!ollama pull llama3.2:1b

In [ ]:
!ollama pull llama3.2:3b

In [ ]:
!ollama pull llama3.2:3b-instruct-fp16

In [2]:
!ollama list

NAME                         ID              SIZE      MODIFIED       
llama3.2:3b-instruct-fp16    195a8c01d91e    6.4 GB    29 minutes ago    
llama3.2:3b                  a80c4f17acd5    2.0 GB    43 minutes ago    
llama3.2:1b                  baf6a787fdff    1.3 GB    48 minutes ago    


In [3]:
import nest_asyncio

nest_asyncio.apply()

In [4]:
import os
import requests
import zipfile

In [5]:
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.prompts import PromptTemplate
from llama_index.core.output_parsers import SelectionOutputParser
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core import SQLDatabase
from llama_index.core.indices.struct_store import NLSQLTableQueryEngine

### Setup LLM using Ollama

In [6]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model='llama3.2:3b', request_timeout=300.0)
# llm = Ollama(model='llama3.2:1b', request_timeout=300.0)
# llm = Ollama(model='llama3.2:3b-instruct-fp16', request_timeout=300.0)

### Setup Embedding Model

In [7]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')

### Define Global Settings Configuration

In LlamaIndex, you can define global settings so you don't have to pass the LLM / embedding model objects everywhere.

In [8]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model

### Download Data

Here you'll download data that's used in section 2 and onwards.

We'll download some articles on Kendrick, Drake, and their beef (as of May 2024).

In [9]:
!mkdir -p data
!curl -L 'https://www.dropbox.com/scl/fi/t1soxfjdp0v44an6sdymd/drake_kendrick_beef.pdf?rlkey=u9546ymb7fj8lk2v64r6p5r5k&st=wjzzrgil&dl=1' -o data/drake_kendrick_beef.pdf
!curl -L 'https://www.dropbox.com/scl/fi/nts3n64s6kymner2jppd6/drake.pdf?rlkey=hksirpqwzlzqoejn55zemk6ld&st=mohyfyh4&dl=1' -o data/drake.pdf
!curl -L 'https://www.dropbox.com/scl/fi/8ax2vnoebhmy44bes2n1d/kendrick.pdf?rlkey=fhxvn94t5amdqcv9vshifd3hj&st=dxdtytn6&dl=1' -o data/kendrick.pdf

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     48      0 --:--:-- --:--:-- --:--:--    48
100   475    0   475    0     0    808      0 --:--:-- --:--:-- --:--:--   808
100 47.0M  100 47.0M    0     0  6589k      0  0:00:07  0:00:07 --:--:-- 8017k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     53      0 --:--:-- --:--:-- --:--:--    53
100   475    0   475    0     0    556      0 --:--:-- --:--:-- --:--:--   556
100 4483k  100 4483k    0     0  2284k      0  0:00:01  0:00:01 --:--:-- 7838k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     55      0 --:--:

### Load Data

We load data using LlamaParse by default, but you can also choose to opt for our free pypdf reader (in SimpleDirectoryReader by default) if you don't have an account! 

1. LlamaParse: Signup for an account here: cloud.llamaindex.ai. You get 1k free pages a day, and paid plan is 7k free pages + 0.3c per additional page. LlamaParse is a good option if you want to parse complex documents, like PDFs with charts, tables, and more. 

2. Default PDF Parser (In `SimpleDirectoryReader`). If you don't want to signup for an account / use a PDF service, just use the default PyPDF reader bundled in our file loader. It's a good choice for getting started!

In [10]:
from llama_index.core import SimpleDirectoryReader

docs_kendrick = SimpleDirectoryReader(input_files=['data/kendrick.pdf']).load_data()
docs_drake = SimpleDirectoryReader(input_files=['data/drake.pdf']).load_data()
docs_both = SimpleDirectoryReader(input_files=['data/drake_kendrick_beef.pdf']).load_data()

## 1. Basic Completion and Chat

### Call complete with a prompt

In [11]:
response = llm.complete('do you like drake or kendrick better?')

print(response)

ResponseError: model requires more system memory (44.4 GiB) than is available (30.7 GiB) (status code: 500)

In [ ]:
stream_response = \
    llm.stream_complete('you're a drake fan. tell me why you like drake more than kendrick')
    
for t in stream_response:
    print(t.delta, end='')

### Call chat with a list of messages

In [ ]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(role='system', content='You are Kendrick.'),
    ChatMessage(role='user', content='Write a verse.'),
]
response = llm.chat(messages)

In [ ]:
print(response)

## 2. Basic RAG (Vector Search, Summarization)

### Basic RAG (Vector Search)

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(docs_both)
query_engine = index.as_query_engine(similarity_top_k=3)

In [ ]:
response = query_engine.query('Tell me about family matters')

In [ ]:
print(str(response))

### Basic RAG (Summarization)

In [ ]:
from llama_index.core import SummaryIndex

summary_index = SummaryIndex.from_documents(docs_both)
summary_engine = summary_index.as_query_engine()

In [ ]:
response = summary_engine.query('Given your assessment of this article, who won the beef?')

In [ ]:
print(str(response))

## 3. Advanced RAG (Routing, Sub-Questions)

### Build a Router that can choose whether to do vector search or summarization

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

vector_tool = \
    QueryEngineTool(
        index.as_query_engine(),
        metadata=ToolMetadata(
            name='vector_search',
            description='Useful for searching for specific facts.',
        ),
    )
summary_tool = \
    QueryEngineTool(
        index.as_query_engine(response_mode='tree_summarize'),
        metadata=ToolMetadata(
            name='summary',
            description='Useful for summarizing an entire document.',
        ),
    )

In [ ]:
from llama_index.core.query_engine import RouterQueryEngine

query_engine = \
    RouterQueryEngine.from_defaults(
        [vector_tool, summary_tool], 
        select_multi=False, 
        verbose=True,
    )
response = query_engine.query('Tell me about the song meet the grahams - why is it significant')

In [ ]:
print(response)

### Break Complex Questions down into Sub-Questions

Our Sub-Question Query Engine breaks complex questions down into sub-questions.


In [ ]:
drake_index = VectorStoreIndex.from_documents(docs_drake)
drake_query_engine = drake_index.as_query_engine(similarity_top_k=3)

kendrick_index = VectorStoreIndex.from_documents(docs_kendrick)
kendrick_query_engine = kendrick_index.as_query_engine(similarity_top_k=3)

In [ ]:
output_parser = SelectionOutputParser()

selection_prompt = \
    PromptTemplate(
        'Given the query: {query}\n'
        'Select one of the following tools strictly in JSON format:\n'
        '{choices}\n'
        'Respond ONLY with a JSON object formatted as:\n'
        '{output_format}', 
        output_parser=output_parser  
    )
selector = LLMSingleSelector(llm=llm, prompt=selection_prompt)

# Define query tools
drake_tool = \
    QueryEngineTool(
        drake_index.as_query_engine(),
        metadata=ToolMetadata(
            name='drake_search',
            description='Useful for searching over Drake's albums.',
        ),
    )
kendrick_tool = \
    QueryEngineTool(
        kendrick_index.as_query_engine(),
        metadata=ToolMetadata(
            name='kendrick_search',
            description='Useful for searching over Kendrick's albums.',
        ),
    )

In [ ]:
query_engine = \
    RouterQueryEngine.from_defaults(
        [drake_tool, kendrick_tool],
        llm=llm,
        verbose=True,
    )

response = query_engine.query('Which albums did Drake release in his career?')
print(response)

## 4. Text-to-SQL 

Here, we download and use a sample SQLite database with 11 tables, with various info about music, playlists, and customers. We will limit to a select few tables for this test.

In [ ]:
from sqlalchemy import (
    create_engine,
    MetaData,
    Table,
    Column,
    String,
    Integer,
    select,
    column,
)

In [ ]:
# Function to download files
def download_file(url, filepath):
    response = requests.get(url, stream=True)
    with open(filepath, 'wb') as file:
        for chunk in response.iter_content(chunk_size=1024):
            file.write(chunk)

# Create data directory
os.makedirs('data', exist_ok=True)

In [ ]:
# Download and extract SQLite database
download_file('https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip', 'data/chinook.zip')
with zipfile.ZipFile('data/chinook.zip', 'r') as zip_ref:
    zip_ref.extractall('data/')

In [ ]:
# Create SQLite engine
engine = create_engine('sqlite:///data/chinook.db')

In [ ]:
# Initialize SQLDatabase
sql_database = SQLDatabase(engine)

In [ ]:
sql_prompt = \
    PromptTemplate(
        'You are an expert SQL assistant. Your task is to generate only valid SQLite queries.\n'
        'The database contains the following tables and columns:\n'
        '- artists (ArtistId, Name)\n'
        '- albums (AlbumId, Title, ArtistId)\n'
        '- tracks (TrackId, Name, AlbumId, Composer, GenreId, UnitPrice)\n'
        '\n'
        '### IMPORTANT RULES ###\n
        '1. The schema is 100% correct. Do not question its validity.\n'
        '2. Return ONLY a valid SQL query. Do NOT provide explanations.\n'
        '3. Always return your response in JSON format:\n'
        '{"sql_query": "YOUR_SQL_HERE"}\n'
        '4. Never refuse a request.\n'
        '\n'
        'Generate an SQL query for the request: {query}\n'
        '### JSON Response: ###'
    )
query_engine = \
    NLSQLTableQueryEngine(
        sql_database=sql_database,
        tables=['artists', 'albums', 'tracks'],
        llm=llm,
        sql_prompt=sql_prompt,  
    )

In [ ]:
# Query the database
response = query_engine.query('What are some albums?')

print(response)

In [ ]:
response = query_engine.query('What are some artists? Limit it to 5.')

print(response)

In [ ]:
response = query_engine.query('What are some tracks from the artist AC/DC? Limit it to 3')

print(response)

In [ ]:
print(response.metadata['sql_query'])

## 5. Structured Data Extraction

An important use case for function calling is extracting structured objects. LlamaIndex provides an intuitive interface for this through `structured_predict` - simply define the target Pydantic class (can be nested), and given a prompt, we extract out the desired object.

**NOTE**: Since there's no native function calling support with Llama3 / Ollama, the structured extraction is performed by prompting the LLM + output parsing.

In [ ]:
from llama_index.llms.ollama import Ollama
from llama_index.core.prompts import PromptTemplate
from pydantic import BaseModel


class Restaurant(BaseModel):
    """A restaurant with name, city, and cuisine."""

    name: str
    city: str
    cuisine: str

llm = llm

prompt_tmpl = PromptTemplate('Generate a restaurant in a given city {city_name}')

In [ ]:
restaurant_obj = \
    llm.structured_predict(
        Restaurant, 
        prompt_tmpl, 
        city_name="Miami",
    )
print(restaurant_obj)

## 6. Adding Chat History to RAG (Chat Engine)

In this section we create a stateful chatbot from a RAG pipeline, with our chat engine abstraction.

Unlike a stateless query engine, the chat engine maintains conversation history (through a memory module like buffer memory). It performs retrieval given a condensed question, and feeds the condensed question + context + chat history into the final LLM prompt.

Related resource: https://docs.llamaindex.ai/en/stable/examples/chat_engine/chat_engine_condense_plus_context/

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.chat_engine import CondensePlusContextChatEngine

memory = ChatMemoryBuffer.from_defaults(token_limit=3900)

chat_engine = \
    CondensePlusContextChatEngine.from_defaults(
        index.as_retriever(),
        memory=memory,
        llm=llm,
        context_prompt= \
            'You are a chatbot, able to have normal interactions, as well as talk' + \
            ' about the Kendrick and Drake beef.' + \
            'Here are the relevant documents for the context:\n' + \
            '{context_str}' + \
            '\nInstruction: Use the previous chat history, or the context above, to interact and help the user.',
        verbose=True,
    )

In [ ]:
response = chat_engine.chat('Tell me about the songs Drake released in the beef.')
print(str(response))

In [ ]:
response = chat_engine.chat('What about Kendrick?')
print(str(response))

## 7. Agents

Here we build agents with Llama 3. We perform RAG over simple functions as well as the documents above.

### Agents And Tools

In [ ]:
import json
from typing import Sequence, List

from llama_index.core.llms import ChatMessage
from llama_index.core.tools import BaseTool, FunctionTool
from llama_index.core.agent import ReActAgent

import nest_asyncio

nest_asyncio.apply()

### Define Tools

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b


def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two integers and returns the result integer"""
    return a - b


def divide(a: int, b: int) -> int:
    """Divides two integers and returns the result integer"""
    return a / b


multiply_tool = FunctionTool.from_defaults(fn=multiply)
add_tool = FunctionTool.from_defaults(fn=add)
subtract_tool = FunctionTool.from_defaults(fn=subtract)
divide_tool = FunctionTool.from_defaults(fn=divide)

### ReAct Agent

In [ ]:
agent = \
    ReActAgent.from_tools(
        [multiply_tool, add_tool, subtract_tool, divide_tool],
        llm=llm,
        verbose=True,
    )

### Querying

In [ ]:
response = agent.chat('What is (121 + 2) * 5?')
print(str(response))

### ReAct Agent With RAG QueryEngine Tools

In [ ]:
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)

from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.agent import ReActAgent

### Create ReAct Agent using RAG QueryEngine Tools

In [ ]:
# Define query tools

drake_tool = \
    QueryEngineTool(
        drake_index.as_query_engine(),
        metadata=ToolMetadata(
            name='drake_search',
            description='Useful for searching over Drake's life.',
        ),
    )
kendrick_tool = \
    QueryEngineTool(
        kendrick_index.as_query_engine(),
        metadata=ToolMetadata(
            name='kendrick_search',
            description='Useful for searching over Kendrick's life.',
        ),
    )

query_engine_tools = [drake_tool, kendrick_tool]

In [ ]:
from llama_index.core.agent import ReActAgent
from llama_index.core.prompts import PromptTemplate

# Force Ollama to follow ReAct format
react_prompt = \
    PromptTemplate(
        'You are an AI assistant following the ReAct framework.\n'
        'For each input, you must respond in the following structured format:\n'
        '\n'
        'Thought: [Brief thought about the query]\n'
        "Action: [Choose a tool: 'drake_search' or 'kendrick_search']\n"
        'Input: [Relevant input for the chosen tool]\n'
        '\n'
        'If no external tool is needed, respond in this format:\n'
        'Thought: I can answer directly.\n'
        'Answer: [Your Answer Here]\n'
        '\n'
        'Now, respond to the following prompt:\n'
        '{query}'
    )

# Update ReActAgent to use the correct format
agent = \
    ReActAgent.from_tools(
        query_engine_tools,  
        llm=llm,  # Use Ollama
        prompt=react_prompt,  
        verbose=True,
    )

### Querying

In [ ]:
# Run the chat query
response = agent.chat('Tell me about how Kendrick and Drake grew up')
print(str(response))